# Demo LLM server: Ollama on a Kaggle GPU, reachable from the local app

Runs Ollama on the Kaggle GPU and publishes it on your fixed ngrok domain, so the locally running
app (Django backend) can use it for answers and summaries during a demo. Everything else (database,
PDFs, embeddings, Qdrant, retrieval) stays on the local machine; only the prompt (question + retrieved
excerpts) is sent here.

**Before running**
1. Notebook settings: *Accelerator* = GPU (T4 ×2 or P100), *Internet* = On (needs a phone-verified Kaggle account).
2. *Add-ons → Secrets*, add and attach to this notebook:
   - `NGROK_AUTHTOKEN`: from https://dashboard.ngrok.com/get-started/your-authtoken
   - `NGROK_DOMAIN`: your free static domain, e.g. `something.ngrok-free.app` (dashboard → Domains)
   - `OLLAMA_API_KEY`: any long random string; the same value goes into `backend/.env`
   - `OLLAMA_MODEL` (optional): defaults to `llama3.2:3b`, the model the app and thesis use
3. For the demo itself use **Save Version → Save & Run All (Commit)**: it runs in the background
   without a browser tab, for at most `MAX_HOURS` (Kaggle stops GPU sessions after 12 h).
   Interactive runs are fine for testing but stop when the session goes idle.

**Backend `.env` on the demo laptop**
```
OLLAMA_BASE_URL=https://<NGROK_DOMAIN>
OLLAMA_API_KEY=<same value as the Kaggle secret>
OLLAMA_FALLBACK_URL=http://localhost:11434   # used automatically if Kaggle is unreachable
OLLAMA_FALLBACK_MODEL=llama3.2:3b
```
GPU time counts against the weekly Kaggle quota (about 30 h).

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def secret(name, default=None):
    try:
        return secrets.get_secret(name)
    except Exception:
        if default is None:
            raise RuntimeError(f"Kaggle secret {name} is missing (Add-ons -> Secrets)")
        return default

NGROK_AUTHTOKEN = secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = secret("NGROK_DOMAIN").replace("https://", "").strip("/")
OLLAMA_API_KEY = secret("OLLAMA_API_KEY")
MODEL = secret("OLLAMA_MODEL", "llama3.2:3b")
MAX_HOURS = 11.5          # stay below Kaggle's 12 h session limit
PROXY_PORT = 8000         # key-checking proxy in front of Ollama (127.0.0.1:11434)
assert len(OLLAMA_API_KEY) >= 16, "use a longer OLLAMA_API_KEY"
print("model:", MODEL, "| domain:", NGROK_DOMAIN)

In [ ]:
# Install and start Ollama on the GPU (same install as eval/kaggle/phase4_llm_generation.ipynb)
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1")  # keep the model in GPU memory
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# Warm-up: load the model once so the first demo question is not slow
import json, time
t0 = time.time()
req = urllib.request.Request("http://127.0.0.1:11434/api/generate",
                             data=json.dumps({"model": MODEL, "prompt": "Say OK.", "stream": False,
                                              "options": {"temperature": 0, "num_ctx": 4096}}).encode(),
                             headers={"Content-Type": "application/json"})
print(json.loads(urllib.request.urlopen(req, timeout=300).read())["response"], f"({time.time() - t0:.1f} s)")

In [ ]:
# Key-checking proxy: only requests with "Authorization: Bearer <OLLAMA_API_KEY>" reach Ollama,
# and only the read/generate endpoints the app uses (no pull/delete/create).
import hmac, threading, urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

ALLOWED = {("POST", "/api/generate"), ("POST", "/api/chat"), ("GET", "/api/tags"),
           ("GET", "/api/ps"), ("GET", "/api/version")}
EXPECTED = f"Bearer {OLLAMA_API_KEY}".encode()
stats = {"ok": 0, "denied": 0, "errors": 0}

class Proxy(BaseHTTPRequestHandler):
    def _handle(self, method):
        auth = (self.headers.get("Authorization") or "").encode()
        if not hmac.compare_digest(auth, EXPECTED):
            stats["denied"] += 1
            return self._reply(401, b'{"error":"unauthorized"}')
        if (method, self.path) not in ALLOWED:
            stats["denied"] += 1
            return self._reply(403, b'{"error":"endpoint not allowed"}')
        body = self.rfile.read(int(self.headers.get("Content-Length") or 0)) if method == "POST" else None
        req = urllib.request.Request(f"http://127.0.0.1:11434{self.path}", data=body, method=method,
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=600) as r:
                stats["ok"] += 1
                return self._reply(r.status, r.read())
        except urllib.error.HTTPError as e:
            stats["errors"] += 1
            return self._reply(e.code, e.read())
        except Exception as e:
            stats["errors"] += 1
            return self._reply(502, json.dumps({"error": str(e)}).encode())

    def _reply(self, status, payload):
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def do_GET(self):
        self._handle("GET")

    def do_POST(self):
        self._handle("POST")

    def log_message(self, fmt, *args):  # keep the notebook output short
        pass

server = ThreadingHTTPServer(("127.0.0.1", PROXY_PORT), Proxy)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("proxy listening on", PROXY_PORT)

In [ ]:
# Publish the proxy on the fixed ngrok domain
!pip -q install pyngrok
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
# newer ngrok agents take the static domain as `url`; older ones as `domain`
try:
    tunnel = ngrok.connect(PROXY_PORT, "http", url=f"https://{NGROK_DOMAIN}")
except Exception as e:
    print("url= option rejected, retrying with domain=:", str(e)[:120])
    tunnel = ngrok.connect(PROXY_PORT, "http", domain=NGROK_DOMAIN)
PUBLIC_URL = tunnel.public_url.replace("http://", "https://")
print("public URL:", PUBLIC_URL)

# self-test through the tunnel: without the key -> 401, with the key -> model list
import requests
print("no key  ->", requests.get(f"{PUBLIC_URL}/api/tags", timeout=30).status_code)
r = requests.get(f"{PUBLIC_URL}/api/tags", headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"}, timeout=30)
print("with key->", r.status_code, [m["name"] for m in r.json().get("models", [])])
print(f"\nbackend/.env:\nOLLAMA_BASE_URL={PUBLIC_URL}\nOLLAMA_MODEL={MODEL}\nOLLAMA_API_KEY=<the secret>")

In [ ]:
# Keep the session alive and print a heartbeat every 5 minutes (stop with the interrupt button)
start = time.time()
try:
    while time.time() - start < MAX_HOURS * 3600:
        alive = ollama_proc.poll() is None
        print(time.strftime("%H:%M:%S"), "ollama", "up" if alive else "DOWN", "| requests", stats, flush=True)
        if not alive:
            break
        time.sleep(300)
finally:
    ngrok.kill()
    server.shutdown()
    print("tunnel closed")